In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

print("TensorFlow Version:", tf.__version__)


TensorFlow Version: 2.20.0


In [2]:
# ─── CONFIGURE PATHS & HYPERPARAMETERS ────────────────────────────────────
IMAGES_DIR = '/Users/shubhbhateja/College/minor_project/JDC_Pipeline/jdc_dataset/skin_images'
CSV_PATH = '/Users/shubhbhateja/College/minor_project/JDC_Pipeline/jdc_dataset/chd_jaundice_jdc.csv'

BATCH_SIZE = 16
EPOCHS = 25
IMG_SIZE = (864, 864)
LEARNING_RATE = 1e-4  # Increased from 1e-5 to escape local minimum

print('Images dir exists:', os.path.isdir(IMAGES_DIR))
print('CSV exists       :', os.path.isfile(CSV_PATH))


Images dir exists: True
CSV exists       : True


In [3]:
df = pd.read_csv(CSV_PATH)
# Ensure we only use rows where the image file actually exists
df['img_path'] = df['image_idx'].apply(lambda x: os.path.join(IMAGES_DIR, str(x)))
df = df[df['img_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Total valid images found: {len(df)}")
print("Class Distribution (Treatment):")
print(df['Treatment'].value_counts())

# Split 80/20 as mentioned in the paper
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Treatment'], random_state=42)
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")

from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.unique(train_df['Treatment']), y=train_df['Treatment'])
class_weight_dict = {0: weights[0], 1: weights[1]}
print("Computed Class Weights:", class_weight_dict)


Total valid images found: 3484
Class Distribution (Treatment):
Treatment
1    2826
0     658
Name: count, dtype: int64
Train samples: 2787, Test samples: 697
Computed Class Weights: {0: np.float64(2.6492395437262357), 1: np.float64(0.6163202122954445)}


In [4]:
def process_path(file_path, label):
    # Load the raw data from the file as a string
    img = tf.io.read_file(file_path)
    # Decode the image
    img = tf.image.decode_jpeg(img, channels=3)
    # Resize the image
    img = tf.image.resize(img, IMG_SIZE)
    # Normalize to [0, 1]
    img = img / 255.0
    return img, label

def prepare_dataset(dataframe, batch_size, shuffle=False):
    paths = dataframe['img_path'].values
    labels = dataframe['Treatment'].values.astype(np.float32)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    if shuffle:
        ds = ds.shuffle(buffer_size=200)
        
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = prepare_dataset(train_df, BATCH_SIZE, shuffle=True)
test_ds = prepare_dataset(test_df, BATCH_SIZE, shuffle=False)


In [5]:
def build_pdf_model():
    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    
    # Block 1
    x = tf.keras.layers.Conv2D(16, (3, 3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    
    # Block 2
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    
    # Block 3
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    
    # Flatten and Dense
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    
    # Output
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="PDF_Architecture_Model")
    return model

model = build_pdf_model()
model.summary()


Model: "PDF_Architecture_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 864, 864, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 862, 862, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 431, 431, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 431, 431, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 429, 429, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 214, 214, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 214, 214, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 212, 212, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 106, 106, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 106, 106, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 719104)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    92,045,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,077,345 (351.25 MB)

 Trainable params: 92,077,345 (351.25 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ─── COMPILE MODEL ────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

# Set up TensorBoard callback
import datetime
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)


In [7]:
%load_ext tensorboard
%tensorboard --logdir logs/fit


In [8]:
# ─── TRAIN MODEL ──────────────────────────────────────────────────────────
print("Starting Training...")
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=[tensorboard_callback]
)


Starting Training...
Epoch 1/25
  5/175 ━━━━━━━━━━━━━━━━━━━━ 7:25 3s/step - accuracy: 0.5671 - auc: 0.5639 - loss: 1.5829 - precision: 0.8950 - recall: 0.5541

KeyboardInterrupt: 

In [ ]:
# ─── EVALUATION & METRICS ────────────────────────────────────────────────
# Predict on test set
print("Evaluating on Test Set...")
preds = model.predict(test_ds)
y_true = test_df['Treatment'].values
y_pred_probs = preds.flatten()
y_pred_classes = (y_pred_probs > 0.5).astype(int)

acc = accuracy_score(y_true, y_pred_classes)
prec = precision_score(y_true, y_pred_classes)
rec = recall_score(y_true, y_pred_classes)
f1 = f1_score(y_true, y_pred_classes)
auc = roc_auc_score(y_true, y_pred_probs)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Model ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Plot Training History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.show()
